# GTEx model building with WGCNA

💡 **Environment:** `clamp-analyses` 

In [1]:
# WGCNA - Following Mantini et al. 2024 Methodology (https://www.nature.com/articles/s41598-024-82563-9#Sec8)

library(WGCNA)
library(here)

enableWGCNAThreads(nThreads = 2)
set.seed(42)

# Load data
gtex_data <- readRDS(here("output/gtex/df_gtex_fbm_filt.rds"))

# WGCNA expects samples x genes
datExpr <- as.data.frame(t(gtex_data))

# Pick soft threshold (R^2 >= 0.9 criterion from paper)
powers <- 1:20
sft <- pickSoftThreshold(datExpr, powerVector = powers, networkType = "unsigned", verbose = 0)

soft_power <- sft$fitIndices$Power[which(sft$fitIndices$SFT.R.sq >= 0.9)[1]]
if (is.na(soft_power)) soft_power <- 7  # Paper's value as fallback

cat("Using soft power:", soft_power, "\n")

# Build network
net <- blockwiseModules(
  datExpr,
  power = soft_power,
  networkType = "unsigned",
  TOMType = "unsigned",
  minModuleSize = 30,
  mergeCutHeight = 0.25,
  numericLabels = TRUE,
  verbose = 3,
  maxBlockSize = ncol(datExpr),
  nThreads = 2
)

# Extract module eigengenes (remove grey/unassigned)
MEs <- net$MEs
if ("ME0" %in% colnames(MEs)) {
  MEs <- MEs[, colnames(MEs) != "ME0"]
}

# Create B matrix (modules x samples)
gtex_wgcna_B <- as.data.frame(t(MEs))
colnames(gtex_wgcna_B) <- rownames(datExpr)

cat("B matrix:", nrow(gtex_wgcna_B), "modules x", ncol(gtex_wgcna_B), "samples\n")

# Save
dir.create(here("output/gtex/wgcna"), showWarnings = FALSE, recursive = TRUE)
write.csv(gtex_wgcna_B, here("output/gtex/wgcna/gtex_wgcna_B.csv"))
saveRDS(gtex_wgcna_B, here("output/gtex/wgcna/gtex_wgcna_B.rds"))
saveRDS(net, here("output/gtex/wgcna/wgcna_network.rds"))

Loading required package: dynamicTreeCut

Loading required package: fastcluster


Attaching package: ‘fastcluster’


The following object is masked from ‘package:stats’:

    hclust





Attaching package: ‘WGCNA’


The following object is masked from ‘package:stats’:

    cor


here() starts at /home/msubirana/Documents/pivlab/clamp-analyses



Allowing parallel execution with up to 2 working processes.
   Power SFT.R.sq  slope truncated.R.sq mean.k. median.k. max.k.
1      1   0.5170  1.470          0.964 6160.00  6160.000  10200
2      2   0.0719 -0.259          0.801 2560.00  2330.000   6010
3      3   0.5500 -0.941          0.837 1270.00  1020.000   3960
4      4   0.6950 -1.260          0.873  705.00   498.000   2770
5      5   0.7430 -1.430          0.889  421.00   261.000   2020
6      6   0.7740 -1.530          0.913  266.00   145.000   1520
7      7   0.7820 -1.620          0.917  175.00    85.900   1170
8      8   0.7840 -1.690          0.924  120.00    53.300    923
9      9   0.7910 -1.730          0.935   83.90    33.800    737
10    10   0.8060 -1.750          0.948   60.20    21.800    595
11    11   0.8180 -1.770          0.959   44.10    14.400    486
12    12   0.8320 -1.770          0.969   32.80     9.610    400
13    13   0.8400 -1.790          0.973   24.80     6.550    332
14    14   0.8500 -1.790      